In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print(train.shape)
print(test.shape)

train.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [3]:
train["text"] = (
    "Question: " + train["prompt"].fillna("") +
    " A: " + train["A"].fillna("") +
    " B: " + train["B"].fillna("") +
    " C: " + train["C"].fillna("") +
    " D: " + train["D"].fillna("") +
    " E: " + train["E"].fillna("")
)

test["text"] = (
    "Question: " + test["prompt"].fillna("") +
    " A: " + test["A"].fillna("") +
    " B: " + test["B"].fillna("") +
    " C: " + test["C"].fillna("") +
    " D: " + test["D"].fillna("") +
    " E: " + test["E"].fillna("")
)

In [4]:
label2id = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

id2label = {v:k for k,v in label2id.items()}

train["label"] = train["answer"].map(label2id)

In [5]:
from transformers import AutoTokenizer

MODEL = "FacebookAI/roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [6]:
import wandb

wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 24f3004935 (24f3004935-dl-genai-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
from datasets import Dataset

train_ds = Dataset.from_pandas(
    train[["text","label"]]
)

test_ds = Dataset.from_pandas(
    test[["text"]]
)

In [8]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [9]:
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=5
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no",
    report_to="wandb"
)

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,3.219861
100,2.462631
150,0.920609
200,0.178952
250,0.022452
300,0.007108
350,0.004834
400,0.003947
450,0.003589
500,0.003316


TrainOutput(global_step=500, training_loss=0.6827297725379468, metrics={'train_runtime': 442.2377, 'train_samples_per_second': 18.09, 'train_steps_per_second': 1.131, 'total_flos': 2104945139712000.0, 'train_loss': 0.6827297725379468, 'epoch': 4.0})

In [12]:
import numpy as np
import torch

pred = trainer.predict(test_ds)

probs = torch.softmax(
    torch.tensor(pred.predictions),
    dim=1
).numpy()

predictions = []

for p in probs:
    top3 = np.argsort(p)[::-1][:3]
    predictions.append(
        " ".join([id2label[i] for i in top3])
    )

In [13]:
submission = pd.DataFrame({
    "id": test["id"],
    "Prediction": predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

,id,Prediction
0,1,A D C
1,2,B C E
2,3,B C D
3,4,E D B
4,5,C D A
